<a href="https://colab.research.google.com/github/monicamtzmdz86-ux/set-up-dashboard-mvp/blob/main/set_up_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

import pandas as pd


df = pd.read_excel(
    '/content/Caso práctico Service centers.xlsx',
    sheet_name='service_centers_dataset.csv'
)

df.head()

,id_sc,nombre_sc,estado,tipo_sc,m2_operativos,throughput_diario_meta,throughput_real_ult_semana,avance_obra_civil_pct,avance_compras_pct,avance_licencias_pct,...,fecha_apertura_real,dias_retraso,nivel_automatizacion,mix_gm_pct,mix_large_pct,mix_oversize_pct,capex_presupuestado_usd,capex_ejercido_usd,estatus_general,observaciones
0,SC-001,SMX-Norte,CDMX,Mediano,1800,3500,3200,100,95,80,...,NaT,0,semi,72,20,8,850000,780000,En operación piloto,Licencias pendientes de validación final
1,SC-002,SMX-Sur,CDMX,Grande,3200,6000,0,85,60,30,...,NaT,35,semi,70,20,10,1250000,620000,En habilitación,Retraso en entrega de equipamiento por proveedor
2,SC-003,GDL-Zapopan,Jalisco,Mediano,2100,4000,0,100,90,95,...,NaT,18,semi,75,18,7,920000,890000,Pre-apertura,Sorter con falla en instalación — reemplazo en...
3,SC-004,MTY-San Nicolás,Nuevo León,Grande,4000,8000,0,60,40,20,...,NaT,0,full,68,22,10,1800000,540000,En obra civil,Sin retrasos previstos a la fecha
4,SC-005,QRO-Centro,Querétaro,Pequeño,750,1500,0,100,100,100,...,2026-02-05,0,manual,78,15,7,380000,395000,Operativo,Apertura completada — sobre presupuesto por aj...


In [5]:

# tamaño
print("Tamaño del dataset:")
print(f"  - Filas: {df.shape[0]}")
print(f"  - Columnas: {df.shape[1]}")
print()

# tipo de datos
print("Tipos de datos por columna:")
print(df.dtypes)
print()

# valores faltantes
print("Valores faltantes por columna:")
print(df.isnull().sum())

Tamaño del dataset:
  - Filas: 12
  - Columnas: 21

Tipos de datos por columna:
id_sc                                 object
nombre_sc                             object
estado                                object
tipo_sc                               object
m2_operativos                          int64
throughput_diario_meta                 int64
throughput_real_ult_semana             int64
avance_obra_civil_pct                  int64
avance_compras_pct                     int64
avance_licencias_pct                   int64
fecha_estimada_apertura       datetime64[ns]
fecha_apertura_real           datetime64[ns]
dias_retraso                           int64
nivel_automatizacion                  object
mix_gm_pct                             int64
mix_large_pct                          int64
mix_oversize_pct                       int64
capex_presupuestado_usd                int64
capex_ejercido_usd                     int64
estatus_general                       object
observaciones       

In [6]:


# Avance total promedio de las 3 categorías

df['avance_total_pct'] = df[
['avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct']
].mean(axis=1).round(1)

# Validar días de retraso
hoy = pd.to_datetime('today').normalize()
df['dias_retraso_calculado'] = (hoy - df['fecha_estimada_apertura']).dt.days

df['dias_retraso_calculado'] = df['dias_retraso_calculado'].clip(lower=0)

# semáforo
def clasificar_retraso(dias):
    if dias < 7:
        return 'Verde (< 7 días)'
    elif dias <= 14:
        return 'Amarillo (7-14 días)'
    else:
        return 'Rojo (> 14 días)'

df['categoria_retraso'] = df['dias_retraso'].apply(clasificar_retraso)

# Desviación de presupuesto (capex)
df['desviacion_capex_pct'] = (
    (df['capex_ejercido_usd'] - df['capex_presupuestado_usd'])
    / df['capex_presupuestado_usd'] * 100
).round(1)

# throughput operativo
df['cumplimiento_throughput_pct'] = (
    df['throughput_real_ult_semana'] / df['throughput_diario_meta'] * 100
).round(1)

#
df[['id_sc', 'estado', 'tipo_sc', 'avance_total_pct',
    'dias_retraso', 'categoria_retraso', 'estatus_general']]

,id_sc,estado,tipo_sc,avance_total_pct,dias_retraso,categoria_retraso,estatus_general
0,SC-001,CDMX,Mediano,91.7,0,Verde (< 7 días),En operación piloto
1,SC-002,CDMX,Grande,58.3,35,Rojo (> 14 días),En habilitación
2,SC-003,Jalisco,Mediano,95.0,18,Rojo (> 14 días),Pre-apertura
3,SC-004,Nuevo León,Grande,40.0,0,Verde (< 7 días),En obra civil
4,SC-005,Querétaro,Pequeño,100.0,0,Verde (< 7 días),Operativo
5,SC-006,Puebla,Mediano,56.7,14,Amarillo (7-14 días),En habilitación
6,SC-007,San Luis Potosí,Pequeño,83.3,8,Amarillo (7-14 días),Pre-apertura
7,SC-008,Baja California,Grande,28.3,0,Verde (< 7 días),En obra civil
8,SC-009,CDMX,Especializado,75.0,21,Rojo (> 14 días),En habilitación
9,SC-010,Veracruz,Pequeño,95.0,5,Verde (< 7 días),Pre-apertura


In [7]:

columnas_orden = [
    'id_sc', 'nombre_sc', 'estado', 'tipo_sc',
    'avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct',
    'avance_total_pct',
    'fecha_estimada_apertura', 'fecha_apertura_real',
    'dias_retraso', 'categoria_retraso',
    'estatus_general',
    'm2_operativos', 'throughput_diario_meta', 'throughput_real_ult_semana',
    'cumplimiento_throughput_pct',
    'capex_presupuestado_usd', 'capex_ejercido_usd', 'desviacion_capex_pct',
    'nivel_automatizacion',
    'mix_gm_pct', 'mix_large_pct', 'mix_oversize_pct',
    'observaciones'
]

df_final = df[columnas_orden]

df_final.to_excel('service_centers_consolidado.xlsx', index=False)

print("✅ Archivo Excel generado exitosamente")
print(f"   Filas: {len(df_final)}")
print(f"   Columnas: {len(df_final.columns)}")

# Vista previa
df_final.head()

✅ Archivo Excel generado exitosamente
   Filas: 12
   Columnas: 25


,id_sc,nombre_sc,estado,tipo_sc,avance_obra_civil_pct,avance_compras_pct,avance_licencias_pct,avance_total_pct,fecha_estimada_apertura,fecha_apertura_real,...,throughput_real_ult_semana,cumplimiento_throughput_pct,capex_presupuestado_usd,capex_ejercido_usd,desviacion_capex_pct,nivel_automatizacion,mix_gm_pct,mix_large_pct,mix_oversize_pct,observaciones
0,SC-001,SMX-Norte,CDMX,Mediano,100,95,80,91.7,2026-02-15,NaT,...,3200,91.4,850000,780000,-8.2,semi,72,20,8,Licencias pendientes de validación final
1,SC-002,SMX-Sur,CDMX,Grande,85,60,30,58.3,2026-04-10,NaT,...,0,0.0,1250000,620000,-50.4,semi,70,20,10,Retraso en entrega de equipamiento por proveedor
2,SC-003,GDL-Zapopan,Jalisco,Mediano,100,90,95,95.0,2026-03-01,NaT,...,0,0.0,920000,890000,-3.3,semi,75,18,7,Sorter con falla en instalación — reemplazo en...
3,SC-004,MTY-San Nicolás,Nuevo León,Grande,60,40,20,40.0,2026-06-20,NaT,...,0,0.0,1800000,540000,-70.0,full,68,22,10,Sin retrasos previstos a la fecha
4,SC-005,QRO-Centro,Querétaro,Pequeño,100,100,100,100.0,2026-01-30,2026-02-05,...,0,0.0,380000,395000,3.9,manual,78,15,7,Apertura completada — sobre presupuesto por aj...


In [8]:
# ============================================
# CELDA 5: Instalar librería de Anthropic
# ============================================

# Instalar el SDK oficial de Anthropic (Claude)
!pip install anthropic --quiet

print("✅ Librería anthropic instalada correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 13.9 MB/s eta 0:00:00
✅ Librería anthropic instalada correctamente


In [9]:
#Ia
import anthropic
import json
from google.colab import userdata

# Cargar la API key
api_key = userdata.get('antropic_api_key')

# Inicializar el cliente de Claude
client = anthropic.Anthropic(api_key=api_key)

# Convertir el DataFrame a JSON para enviarlo a Claude
# Solo enviamos las columnas más relevantes
columnas_para_ia = [
    'id_sc', 'nombre_sc', 'estado', 'tipo_sc',
    'avance_obra_civil_pct', 'avance_compras_pct', 'avance_licencias_pct',
    'avance_total_pct', 'dias_retraso', 'categoria_retraso',
    'estatus_general', 'desviacion_capex_pct', 'observaciones'
]


df_para_ia = df[columnas_para_ia].copy()
df_json = df_para_ia.to_json(orient='records', force_ascii=False)


def generar_resumen_ejecutivo(datos_json: str) -> str:
    """
    Envía los datos de Service Centers a Claude y obtiene
    un resumen ejecutivo con riesgos y acciones recomendadas.
    """

    prompt = f"""Eres un analista senior de operaciones logísticas especializado
en apertura de Service Centers. Analiza el siguiente reporte semanal de aperturas
y genera un análisis ejecutivo estructurado.

DATOS DE LOS 12 SERVICE CENTERS:
{datos_json}

INSTRUCCIONES:

Genera tu respuesta en este formato exacto en español:

## RESUMEN EJECUTIVO
Un párrafo de máximo 80 palabras con la situación general de las aperturas,
mencionando porcentaje promedio de avance, número de SCs en riesgo crítico,
y el estado más afectado.

## TOP 3 RIESGOS OPERATIVOS DE LA SEMANA
Lista los 3 riesgos más urgentes basados en los datos. Para cada uno:
- *Riesgo:* descripción clara en una línea
- *SCs afectados:* IDs específicos
- *Impacto:* consecuencia operativa o financiera

## ACCIONES RECOMENDADAS
Por cada riesgo identificado arriba, propón una acción concreta y accionable:
- Qué hacer
- Quién debe ejecutarla (perfil del responsable)
- Plazo sugerido (días)

Usa lenguaje ejecutivo: directo, sin tecnicismos innecesarios, orientado a acción.
"""

    # Llamada a la API de Claude
    mensaje = client.messages.create(
        model='claude-sonnet-4-5',  # Modelo actual de Sonnet
        max_tokens=1500,
        messages=[
            {'role': 'user', 'content': prompt}
        ]
    )

    return mensaje.content[0].text


# Ejecutar el análisis
print("🤖 Enviando datos a Claude para análisis...\n")
print("=" * 70)

resumen = generar_resumen_ejecutivo(df_json)
print(resumen)

print("=" * 70)
print("\n✅ Análisis completado exitosamente")

🤖 Enviando datos a Claude para análisis...

## RESUMEN EJECUTIVO

El portafolio de 12 Service Centers presenta un avance promedio de 65.3%. Se identifican 3 SCs en estado crítico (rojo) con retrasos superiores a 14 días: SC-002 (35 días), SC-003 (18 días) y SC-009 (21 días). CDMX es el estado más afectado con 2 de 3 casos críticos. Solo 1 SC está operativo. La desviación CAPEX promedio es favorable (-36.5%), aunque 5 proyectos presentan ejecución presupuestal inferior al 50%.

## TOP 3 RIESGOS OPERATIVOS DE LA SEMANA

**1. RIESGO CRÍTICO DE APERTURA POR DEPENDENCIA DE PROVEEDORES**
- **Riesgo:** Retrasos severos por incumplimiento de proveedores de equipamiento crítico (sorters, mesas clasificación)
- **SCs afectados:** SC-002 (35 días), SC-003 (18 días), SC-007 (8 días)
- **Impacto:** Postergación de aperturas comerciales, pérdida de capacidad operativa estimada en 3-5 semanas, penalizaciones contractuales potenciales

**2. BLOQUEO REGULATORIO EN LICENCIAS Y PERMISOS**
- **Riesgo:** T

In [12]:
from datetime import datetime

# Guardar el análisis en un archivo de texto
fecha_hoy = datetime.now().strftime('%Y-%m-%d')
Resumen = f'resumen_ejecutivo_{fecha_hoy}.md'

with open(Resumen, 'w', encoding='utf-8') as f:
    f.write(f"# Reporte Ejecutivo de Aperturas - Service Centers\n")
    f.write(f"*Fecha de generación:* {fecha_hoy}\n")
    f.write(f"*Generado por:* Módulo de IA (Claude API)\n\n")
    f.write("---\n\n")
    f.write(resumen)

print(f"✅ Resumen guardado en: {nombre_archivo}")

✅ Resumen guardado en: resumen_ejecutivo_2026-05-24.md
